
# V36 TARGET595 — Exact Reproduction Notebook

This notebook reproduces the submission that scored **0.606** on the public leaderboard:

`submission_v36_TARGET595_rerank.csv`

It does **not retrain the expensive transformer models**. Instead, it reuses their saved LIVE-split
OOF/test probabilities and rebuilds the exact reranking logic that produced the 0.606 submission.

## Components

- V14 protected champion baseline
- V35 transliteration-XLM-R toxic reranking
- V36 regularized meta reranking for:
  - `implicit_toxicity`
  - `contains_profanity`
- V35 BGE threatening reranking
- every changed label keeps the **same positive count as V14**

## Offline validation

- V14 validation: **0.63444**
- V36 TARGET595 validation: **0.64719**
- validation gain: **+0.01275**

## Public leaderboard result

**0.606**

Expected final CSV SHA-256:

`4d8f13d9a3a89b1685345cd0d4ca5d1c3c2ec935126448ebfac345e2f5188260`

This notebook uses only the Kaggle-accepted LIVE ID set beginning:

`BICH_0003, BICH_0006, BICH_0015, BICH_0016`


In [ ]:

# 1. Upload the compact V36 reproduction data bundle
from google.colab import files
import pathlib, zipfile

uploaded = files.upload()
assert "v36_TARGET595_reproduce_data.zip" in uploaded,     "Upload v36_TARGET595_reproduce_data.zip"

DATA_DIR = pathlib.Path("/content/v36_target595")
DATA_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile("v36_TARGET595_reproduce_data.zip") as z:
    z.extractall(DATA_DIR)

print("Extracted:")
for p in sorted(DATA_DIR.iterdir()):
    print(" ", p.name, f"{p.stat().st_size:,} bytes")


In [ ]:

# 2. Imports + strict LIVE split verification
import re, json, hashlib
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score, roc_auc_score

SEED = 2026

LABELS = [
    "is_toxic",
    "implicit_toxicity",
    "sarcasm",
    "contains_profanity",
    "threatening",
    "identity_attack",
    "sexual_content",
]
LI = {l:i for i,l in enumerate(LABELS)}
ATTRS = LABELS[1:]

train = pd.read_csv(DATA_DIR/"train.csv", dtype={"ID":str})
valid = pd.read_csv(DATA_DIR/"validation.csv", dtype={"ID":str})
test  = pd.read_csv(DATA_DIR/"test.csv", dtype={"ID":str})
v2v   = pd.read_csv(DATA_DIR/"v2_valid.csv", dtype={"ID":str})
v2t   = pd.read_csv(DATA_DIR/"v2_test.csv", dtype={"ID":str})
champ = pd.read_csv(DATA_DIR/"submission_v14.csv", dtype={"ID":str})

assert len(train) == 3297
assert len(valid) == 500
assert len(test) == 1000
assert test["ID"].tolist() == champ["ID"].tolist()
assert test["ID"].tolist() == v2t["ID"].tolist()
assert valid["ID"].tolist() == v2v["ID"].tolist()
assert test["ID"].head(4).tolist() == [
    "BICH_0003", "BICH_0006", "BICH_0015", "BICH_0016"
]

NTR = len(train)
NVAL = len(valid)
VIDX = np.arange(NTR, NTR+NVAL)

print("✅ LIVE split verified:", len(train), len(valid), len(test))
print("First IDs:", test["ID"].head(6).tolist())


In [ ]:

# 3. Load all saved OOF/test probabilities

oldb = np.load(DATA_DIR/"oldb.npz")
oldx = np.load(DATA_DIR/"oldx.npz")

v17b = np.load(DATA_DIR/"v17b.npz")
v17x = np.load(DATA_DIR/"v17x.npz")
v18  = np.load(DATA_DIR/"v18.npz")
v19  = np.load(DATA_DIR/"v19.npz")
v23  = np.load(DATA_DIR/"v23.npz")
v25  = np.load(DATA_DIR/"v25.npz")
v32  = np.load(DATA_DIR/"v32.npz")
v33  = np.load(DATA_DIR/"v33.npz")
v35  = np.load(DATA_DIR/"v35.npz")
v15  = np.load(DATA_DIR/"v15.npz")

fold = np.load(DATA_DIR/"fold_assignment.npy")
assert fold.shape == (3797,)
assert (fold >= 0).all()

ftr = fold[:NTR]
fva = fold[VIDX]

print("✅ probability files loaded")
print("Fold counts:", np.bincount(fold))


In [ ]:

# 4. Reconstruct exact V14 validation/test probability family

def logit(p):
    p = np.clip(np.asarray(p,float), 1e-5, 1-1e-5)
    return np.log(p/(1-p))

def sigmoid(z):
    return 1/(1+np.exp(-z))

v14v = {}
v14t = {}

v14v["is_toxic"] = (
    .875*v2v["is_toxic_prob"].to_numpy(float)
    + .125*oldx["valid"][1:3,:,0].mean(0)
)
v14t["is_toxic"] = (
    .875*v2t["is_toxic_prob"].to_numpy(float)
    + .125*oldx["test"][1:3,:,0].mean(0)
)

j = LI["implicit_toxicity"]
v14v["implicit_toxicity"] = sigmoid(
    .9*logit(v2v["implicit_toxicity_prob"].to_numpy(float))
    + .1*logit(oldb["valid"].mean(0)[:,j])
)
v14t["implicit_toxicity"] = sigmoid(
    .9*logit(v2t["implicit_toxicity_prob"].to_numpy(float))
    + .1*logit(oldb["test"].mean(0)[:,j])
)

v14v["sarcasm"] = v2v["sarcasm_prob"].to_numpy(float)
v14t["sarcasm"] = v2t["sarcasm_prob"].to_numpy(float)

j = LI["contains_profanity"]
v14v["contains_profanity"] = (
    .95*v2v["contains_profanity_prob"].to_numpy(float)
    + .05*oldb["valid"][0,:,j]
)
v14t["contains_profanity"] = (
    .95*v2t["contains_profanity_prob"].to_numpy(float)
    + .05*oldb["test"][0,:,j]
)

j = LI["threatening"]
v14v["threatening"] = (
    .95*v2v["threatening_prob"].to_numpy(float)
    + .05*oldx["valid"][0,:,j]
)
v14t["threatening"] = (
    .95*v2t["threatening_prob"].to_numpy(float)
    + .05*oldx["test"][0,:,j]
)

j = LI["identity_attack"]
v14v["identity_attack"] = (
    .70*v2v["identity_attack_prob"].to_numpy(float)
    + .30*oldb["valid"][2,:,j]
)
v14t["identity_attack"] = (
    .70*v2t["identity_attack_prob"].to_numpy(float)
    + .30*oldb["test"][2,:,j]
)

v14v["sexual_content"] = v2v["sexual_content_prob"].to_numpy(float)
v14t["sexual_content"] = v2t["sexual_content_prob"].to_numpy(float)

TH = {
    "is_toxic": .6850110311061144,
    "implicit_toxicity": .1416,
    "sarcasm": .175,
    "contains_profanity": .245,
    "threatening": .16558668018085876,
    "identity_attack": .3814,
    "sexual_content": .245,
}

v14pred = np.column_stack([
    (v14v[l] >= TH[l]).astype(int)
    for l in LABELS
])

v14test = np.column_stack([
    (v14t[l] >= TH[l]).astype(int)
    for l in LABELS
])

assert np.array_equal(v14test, champ[LABELS].to_numpy(int))
print("✅ V14 reconstructed exactly")


In [ ]:

# 5. Build the train-OOF source matrix used by V36 meta reranking

all_sources = {l:[] for l in LABELS}
test_sources = {l:[] for l in LABELS}

def add(label, oof_prob, test_prob):
    all_sources[label].append(np.asarray(oof_prob,float))
    test_sources[label].append(np.asarray(test_prob,float))

# Multi-label models
for oo,tt in [
    (v18["oof"], v18["test"]),
    (v32["stage2_oof"], v32["stage2_test"]),
    (v32["lexical_oof"], v32["lexical_test"]),
    (v33["me5_oof"], v33["me5_test"]),
    (v33["twitter_oof"], v33["twitter_test"]),
    (v35["e5_oof"], v35["e5_test"]),
    (v35["bge_oof"], v35["bge_test"]),
    (v35["tbxlmr_oof"], v35["tbxlmr_test"]),
]:
    for j,l in enumerate(LABELS):
        add(l, oo[:,j], tt[:,j])

# Conditional V23 attributes
for j,l in enumerate(ATTRS):
    add(l, v23["oof"][:,j], v23["test"][:,j])

# V15 rare-label specialists
for j,l in enumerate([
    "implicit_toxicity",
    "contains_profanity",
    "threatening",
    "sexual_content",
]):
    add(l, v15["oof"][:,j], v15["test"][:,j])

# Dedicated toxic models
for arr in [v17b, v17x, v19, v25]:
    add("is_toxic", arr["oof"], arr["test"])

add("is_toxic", v32["stage1_oof"], v32["stage1_test"])

TOX_O = list(all_sources["is_toxic"])
TOX_T = list(test_sources["is_toxic"])

print("OOF source counts:")
for l in LABELS:
    print(l, len(all_sources[l]))


In [ ]:

# 6. Add simple text-meta features

BN = re.compile(r"[\u0980-\u09FF]")
LAT = re.compile(r"[A-Za-z]")

def meta(df):
    rows = []

    for s in df["text"].astype(str):
        n = max(len(s),1)
        lat = sum(c.isascii() and c.isalpha() for c in s)
        bn = sum("\u0980" <= c <= "\u09FF" for c in s)
        words = len(s.split())

        rows.append([
            np.log1p(n),
            np.log1p(words),
            bn/n,
            lat/n,
            np.log1p(s.count("!")),
            np.log1p(s.count("?")),
            int(bool(BN.search(s)) and not bool(LAT.search(s))),
            int(bool(BN.search(s)) and bool(LAT.search(s))),
            int(bool(LAT.search(s)) and not bool(BN.search(s))),
        ])

    return np.asarray(rows,float)

combined = pd.concat([train,valid],ignore_index=True)
M = meta(combined)
MT = meta(test)

print("Meta feature shape:", M.shape, MT.shape)


In [ ]:

# 7. Exact competition metric helpers

def f1_positive(y,p):
    y = np.asarray(y,int)
    p = np.asarray(p,int)

    tp = ((y==1)&(p==1)).sum()
    fp = ((y==0)&(p==1)).sum()
    fn = ((y==1)&(p==0)).sum()

    d = 2*tp + fp + fn
    return float(2*tp/d) if d else 0.0

def metric(label,y,p):
    if label == "is_toxic":
        vals = []
        for c in [0,1]:
            vals.append(
                f1_positive(
                    (y==c).astype(int),
                    (p==c).astype(int)
                )
            )
        return float(np.mean(vals))

    return f1_positive(y,p)

def topk(prob,k):
    out = np.zeros(len(prob),int)
    k = max(0,min(int(k),len(prob)))

    if k:
        out[np.argsort(-prob)[:k]] = 1

    return out

def official(y,pred):
    scores = [
        metric(l,y[:,j],pred[:,j])
        for j,l in enumerate(LABELS)
    ]

    final = .5*scores[0] + .5*np.mean(scores[1:])
    return float(final), scores

ytr = train[LABELS].to_numpy(int)
yva = valid[LABELS].to_numpy(int)

print("Metric helpers ready")


In [ ]:

# 8. Train the fast regularized V36 meta-rerankers
#
# This step is cheap: only small logistic-regression stackers are fit.
# C is selected using TRAIN OOF folds, not the 500-row validation labels.

chosen = {}
report = []

for j,label in enumerate(LABELS):
    cols = list(all_sources[label])
    tcols = list(test_sources[label])

    # Secondary labels also receive toxic-context probabilities.
    if label != "is_toxic":
        cols += TOX_O
        tcols += TOX_T

    X = np.column_stack(cols)
    Xt = np.column_stack(tcols)

    X = np.column_stack([
        X,
        logit(X),
        M,
    ])

    Xt = np.column_stack([
        Xt,
        logit(Xt),
        MT,
    ])

    Xtr = X[:NTR]
    Xv = X[VIDX]

    best_cv = None

    for C in [.01,.03,.1,.3]:
        oof = np.zeros(NTR,float)

        for f in range(5):
            tr_mask = ftr != f
            va_mask = ftr == f

            mod = Pipeline([
                ("scale", StandardScaler()),
                ("lr", LogisticRegression(
                    C=C,
                    solver="liblinear",
                    max_iter=700,
                    random_state=SEED+f,
                ))
            ])

            mod.fit(
                Xtr[tr_mask],
                ytr[tr_mask,j]
            )

            oof[va_mask] = mod.predict_proba(
                Xtr[va_mask]
            )[:,1]

        if label == "is_toxic":
            rank_score = roc_auc_score(
                ytr[:,j],
                oof
            )
        else:
            rank_score = average_precision_score(
                ytr[:,j],
                oof
            )

        if best_cv is None or rank_score > best_cv[0]:
            best_cv = (rank_score,C)

    C = best_cv[1]

    final_model = Pipeline([
        ("scale", StandardScaler()),
        ("lr", LogisticRegression(
            C=C,
            solver="liblinear",
            max_iter=1000,
            random_state=SEED,
        ))
    ])

    final_model.fit(
        Xtr,
        ytr[:,j]
    )

    pv = final_model.predict_proba(Xv)[:,1]
    pt = final_model.predict_proba(Xt)[:,1]

    base = v14pred[:,j]
    base_score = metric(
        label,
        yva[:,j],
        base
    )

    kv = int(base.sum())
    kt = int(champ[label].sum())

    candidates = []

    for w in [0,.2,.4,.6,.8,1.0]:
        sv = (
            (1-w)*logit(v14v[label])
            + w*logit(pv)
        )

        st = (
            (1-w)*logit(v14t[label])
            + w*logit(pt)
        )

        vp = topk(sv,kv)
        tp = topk(st,kt)

        score = metric(
            label,
            yva[:,j],
            vp
        )

        fold_scores = []
        base_fold_scores = []

        for f in range(5):
            mask = fva == f
            kk = int(base[mask].sum())

            fold_scores.append(
                metric(
                    label,
                    yva[mask,j],
                    topk(sv[mask],kk)
                )
            )

            base_fold_scores.append(
                metric(
                    label,
                    yva[mask,j],
                    base[mask]
                )
            )

        fs = np.asarray(fold_scores)
        bs = np.asarray(base_fold_scores)

        wins = int((fs > bs + 1e-12).sum())
        losses = int((fs < bs - 1e-12).sum())
        changes = int(
            (tp != champ[label].to_numpy(int)).sum()
        )

        candidates.append({
            "score":score,
            "w":w,
            "wins":wins,
            "losses":losses,
            "changes":changes,
            "val_pred":vp,
            "test_pred":tp,
            "fold_scores":fold_scores,
            "base_fold_scores":base_fold_scores,
        })

    best = max(
        candidates,
        key=lambda x:(
            x["score"],
            x["wins"],
            -x["losses"],
            -x["changes"],
        )
    )

    chosen[label] = best

    report.append({
        "label":label,
        "train_cv_rank":best_cv[0],
        "C":C,
        "v14_validation":base_score,
        "best_validation":best["score"],
        "gain":best["score"]-base_score,
        "blend_weight":best["w"],
        "wins":best["wins"],
        "ties":5-best["wins"]-best["losses"],
        "losses":best["losses"],
        "test_changes":best["changes"],
    })

report_df = pd.DataFrame(report)
display(report_df.round(6))


In [ ]:

# 9. Build the exact V36 TARGET595 components
#
# Component 1: toxic
# V14 + transliteration-XLM-R logit blend 70/30,
# preserve V14 positive counts.

tb_val = v35["tbxlmr_oof"][VIDX, LI["is_toxic"]]
tb_test = v35["tbxlmr_test"][:, LI["is_toxic"]]

toxic_val_score = sigmoid(
    .70*logit(v14v["is_toxic"])
    + .30*logit(tb_val)
)

toxic_test_score = sigmoid(
    .70*logit(v14t["is_toxic"])
    + .30*logit(tb_test)
)

tox_v = topk(
    toxic_val_score,
    int(v14pred[:,LI["is_toxic"]].sum())
)

tox_t = topk(
    toxic_test_score,
    int(champ["is_toxic"].sum())
)

# Component 2: threatening
# BGE cross-script score, preserving V14 positive count.

j = LI["threatening"]

thr_v = topk(
    v35["bge_oof"][VIDX,j],
    int(v14pred[:,j].sum())
)

thr_t = topk(
    v35["bge_test"][:,j],
    int(champ["threatening"].sum())
)

print("Components ready")


In [ ]:

# 10. Assemble V36 SAFE and V36 TARGET595

safe_val = v14pred.copy()
safe_test = champ[LABELS].to_numpy(int).copy()

# toxic component
safe_val[:,LI["is_toxic"]] = tox_v
safe_test[:,LI["is_toxic"]] = tox_t

# meta-reranked implicit + profanity
for label in [
    "implicit_toxicity",
    "contains_profanity",
]:
    safe_val[:,LI[label]] = chosen[label]["val_pred"]
    safe_test[:,LI[label]] = chosen[label]["test_pred"]

# TARGET595 = SAFE + BGE threat reranking
target_val = safe_val.copy()
target_test = safe_test.copy()

target_val[:,LI["threatening"]] = thr_v
target_test[:,LI["threatening"]] = thr_t

base_score, base_scores = official(
    yva,
    v14pred
)

safe_score, safe_scores = official(
    yva,
    safe_val
)

target_score, target_scores = official(
    yva,
    target_val
)

print("V14 validation:", base_score)
print("V36 SAFE validation:", safe_score)
print("V36 TARGET595 validation:", target_score)
print("TARGET595 gain:", target_score-base_score)

print()
print("Per-label TARGET595 validation:")
for l,s in zip(LABELS,target_scores):
    print(f"{l:22s} {s:.6f}")


In [ ]:

# 11. Create the exact V36 TARGET595 submission + verify hash

submission = champ.copy()

for j,label in enumerate(LABELS):
    submission[label] = target_test[:,j].astype(int)

assert submission["ID"].tolist() == test["ID"].tolist()
assert submission["ID"].tolist() == champ["ID"].tolist()
assert submission["ID"].head(4).tolist() == [
    "BICH_0003",
    "BICH_0006",
    "BICH_0015",
    "BICH_0016",
]
assert submission[LABELS].isin([0,1]).all().all()

OUT = "/content/submission_v36_TARGET595_rerank.csv"
submission.to_csv(OUT,index=False)

with open(OUT,"rb") as f:
    actual_sha = hashlib.sha256(f.read()).hexdigest()

EXPECTED_SHA = "4d8f13d9a3a89b1685345cd0d4ca5d1c3c2ec935126448ebfac345e2f5188260"

print("Expected SHA-256:", EXPECTED_SHA)
print("Actual SHA-256:  ", actual_sha)

assert actual_sha == EXPECTED_SHA, (
    "Output differs from the exact 0.606 submission."
)

print("\n✅ Exact V36 TARGET595 submission reproduced.")
print("✅ Public leaderboard result for this exact CSV: 0.606")


In [ ]:

# 12. Save report and download final CSV

report_df.to_csv(
    "/content/v36_target595_meta_report.csv",
    index=False
)

summary = {
    "public_leaderboard_score": 0.606,
    "protected_previous_champion": 0.586,
    "offline_v14_validation": float(base_score),
    "offline_target595_validation": float(target_score),
    "offline_gain": float(target_score-base_score),
    "total_test_prediction_changes": int(
        (
            target_test
            != champ[LABELS].to_numpy(int)
        ).sum()
    ),
    "components": {
        "is_toxic":
            "V14 + TB-XLM-R logit blend 70/30; fixed V14 positive count",
        "implicit_toxicity":
            "V36 regularized meta rerank; fixed V14 positive count",
        "contains_profanity":
            "V36 regularized meta rerank; fixed V14 positive count",
        "threatening":
            "V35 BGE rerank; fixed V14 positive count",
        "sarcasm":
            "unchanged V14",
        "identity_attack":
            "unchanged V14",
        "sexual_content":
            "unchanged V14",
    }
}

with open(
    "/content/v36_TARGET595_summary.json",
    "w"
) as f:
    json.dump(summary,f,indent=2)

print(json.dumps(summary,indent=2))

from google.colab import files
files.download(
    "/content/submission_v36_TARGET595_rerank.csv"
)
